# Pydantic model to structure output

In [3]:
#TODO:skapa en agent som ska simulera en IT-anställd
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

employee_simulator_agent = Agent(
    "openrouter:liquid/lfm-2.5-1.2b-thinking:free",
    system_prompt="""
    You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.

Fields to include in output:
- name
- age
- gender
- job_title
- salary in SEK per month
    """
)



result = await employee_simulator_agent.run("simulate two employees")

result

AgentRunResult(output='Here are two simulated employee profiles as per your request:\n\n**Employee 1:**  \n- **Name:** Erik Malle  \n- **Age:** 35  \n- **Gender:** Male  \n- **Job Title:** Data Engineer  \n- **Salary:** 480,000 SEK/month  \n\n**Employee 2:**  \n- **Name:** Anna Lund  \n- **Age:** 28  \n- **Gender:** Female  \n- **Job Title:** Machine Learning Specialist  \n- **Salary:** 450,000 SEK/month  \n\nThese roles align with Sweden’s IT/data science landscape, reflecting diverse professional contributions. Let me know if you need adjustments!')

In [4]:
print(result.output)

Here are two simulated employee profiles as per your request:

**Employee 1:**  
- **Name:** Erik Malle  
- **Age:** 35  
- **Gender:** Male  
- **Job Title:** Data Engineer  
- **Salary:** 480,000 SEK/month  

**Employee 2:**  
- **Name:** Anna Lund  
- **Age:** 28  
- **Gender:** Female  
- **Job Title:** Machine Learning Specialist  
- **Salary:** 450,000 SEK/month  

These roles align with Sweden’s IT/data science landscape, reflecting diverse professional contributions. Let me know if you need adjustments!


In [5]:
with open('simulated_employees.md', 'w') as file:
    file.write(result.output)

## Get more structured output

issue with above:

 - output structure vary    
 - hard to work with the data e.g. compute mean of salaries

 want:
    - repeatable structure

In [12]:
from pydantic import BaseModel, Field
from typing import Literal

class employeeModel(BaseModel):
    name: str = Field(description='Mostly swedish names, but could be foreign names as well')
    age: int = Field(description= 'age should be between 18 and 67')
    gender: Literal['Male','Female']
    experience_level: Literal['Entry', 'Mid level', 'Senior', 'Expert']
    job_title: str
    salary: int = Field(gte=30_000, lte=50_000, description='salary shouldb be between 30k and 50k, the higher experience level, the higher salary')

employee_simulator_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="""
    You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.""")

result= await employee_simulator_agent.run('Give me three employees', output_type=list[employeeModel])
result

C:\Users\Lilit\AppData\Local\Temp\ipykernel_21784\2176293892.py:10: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte', 'lte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  salary: int = Field(gte=30_000, lte=50_000, description='salary shouldb be between 30k and 50k, the higher experience level, the higher salary')


AgentRunResult(output=[employeeModel(name='Anna Svensson', age=32, gender='Female', experience_level='Senior', job_title='Data Scientist', salary=45000), employeeModel(name='David Johansson', age=25, gender='Male', experience_level='Entry', job_title='Machine Learning Engineer', salary=32000), employeeModel(name='Emma Eriksson', age=38, gender='Female', experience_level='Mid level', job_title='Data Engineer', salary=39000)])

In [13]:
result.output

[employeeModel(name='Anna Svensson', age=32, gender='Female', experience_level='Senior', job_title='Data Scientist', salary=45000),
 employeeModel(name='David Johansson', age=25, gender='Male', experience_level='Entry', job_title='Machine Learning Engineer', salary=32000),
 employeeModel(name='Emma Eriksson', age=38, gender='Female', experience_level='Mid level', job_title='Data Engineer', salary=39000)]